# 逐层点时账户分钟高水位回撤余量去重

冻结 `plan.json` 后，只读 2018–2020 父本自身账户与双 ETF Qlib 分钟收盘；保留 2021/2022。核对开盘 drawdown 恒等式和分钟变动，不读取未来动作价值或训练模型。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/"research/stage1_v5_minute_cushion_overlap_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
for key in ("parent","frame"):
 s=plan[key];assert sha(root/s["card"]/"summary.json")==s["summary_sha256"] and sha(root/s["card"]/"verification_manifest.json")==s["manifest_sha256"]
 assert json.loads((root/s["card"]/"nas_binding.json").read_text())["evidence"]["sha256"]==s["nas_archive_sha256"]
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
def load(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s["card"],expected_recorder_id=s["recorder_id"],artifacts=[name],expected_manifest_sha256=s["manifest_sha256"])
 return v[name]
book=load("parent","analysis/native/dated_f696_0")
frame=load("frame","inputs/covered_frame").reset_index(drop=True)
assert len(frame)==len(book["targets"])==1056
assert pd.to_datetime(frame.trade_date).equals(pd.to_datetime(book["targets"].trade_date).reset_index(drop=True))
source=D.features(["SOXL","SOXX"],["$close","$price_valid"],start_time="2018-10-15 09:30:00",end_time="2020-12-31 16:00:00",freq="1min")
soxl=source.xs("SOXL",level="instrument");soxx=source.xs("SOXX",level="instrument")
print('verified source and parent; minute rows',len(source))

[20825:MainThread](2026-10-30 13:21:30,295) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[20825:MainThread](2026-10-30 13:21:30,731) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[20825:MainThread](2026-10-30 13:21:30,732) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


verified source and parent; minute rows 432720


In [2]:
import warnings
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
# Same peak recursion as stage1_event_episode: update at known open and at actual close.
def quant(pos):
 return np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ("SOXL","SOXX")],float)
initial=float(book["targets"].iloc[0].known_equity);peak=initial;out=[];max_open_err=0.;missing=0
for i,row in frame.iterrows():
 day=pd.Timestamp(row.trade_date)
 if day.year not in plan["discovery_years"]:break
 target=book["targets"].iloc[i]
 known=float(target.known_equity);peak=max(peak,known)
 dd_open=max(0.,1-known/peak)
 max_open_err=max(max_open_err,abs(dd_open-float(target.known_drawdown)))
 t0=day+pd.Timedelta(hours=9,minutes=47);tr=day+pd.Timedelta(hours=10,minutes=32);tc=max(k for k in book["native_positions"] if k.normalize()==day)
 before=book["native_positions"].get(t0);after=book["native_positions"].get(tr);closing=book["native_positions"].get(tc)
 assert before is not None and after is not None and closing is not None
 x=soxl.loc[soxl.index.normalize()==day];y=soxx.loc[soxx.index.normalize()==day]
 pair=x[["$close","$price_valid"]].join(y[["$close","$price_valid"]],lsuffix="_soxl",rsuffix="_soxx",how="inner")
 pair=pair.loc[(pair.index>=t0+pd.Timedelta(minutes=1)) & (pair["$price_valid_soxl"]==1) & (pair["$price_valid_soxx"]==1)]
 if pair.empty:missing+=1;continue
 early=pair.index<tr
 q0,q1=quant(before),quant(after);cash0,cash1=float(before.get_cash()),float(after.get_cash())
 prices=pair[["$close_soxl","$close_soxx"]].to_numpy(float)
 nav=np.empty(len(pair));nav[early]=prices[early]@q0+cash0;nav[~early]=prices[~early]@q1+cash1
 dds=np.maximum(0.,1-nav/peak)
 assert np.isfinite(dds).all() and np.min(nav)>0
 out.append({"trade_date":str(day.date()),"year":int(day.year),"risk_state":int(book["independent_replay"].iloc[i].risk_state),"recovery_action":bool(book["independent_replay"].iloc[i].recovery_action),"minute_bars":len(pair),"open_drawdown":dd_open,"minute_drawdown_min":float(dds.min()),"minute_drawdown_max":float(dds.max()),"minute_drawdown_range":float(np.ptp(dds)),"minute_end_drawdown":float(dds[-1])})
 closeq=quant(closing);closecash=float(closing.get_cash());closeprice=row[["close_soxl","close_soxx"]].to_numpy(float)
 peak=max(peak,float(closeq@closeprice+closecash))
assert max_open_err<1e-8 and missing==0
z=pd.DataFrame(out)
metrics={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"days":len(z),"source_minute_rows":len(source),"max_parent_open_drawdown_error":max_open_err,"missing_pair_days":missing,"recovery_days":int(z.recovery_action.sum()),"minute_bars":int(z.minute_bars.sum()),"days_with_nonzero_minute_drawdown_range":int(z.minute_drawdown_range.gt(1e-10).sum()),"median_intraday_drawdown_range_pct_points":float(100*z.minute_drawdown_range.median()),"p90_intraday_drawdown_range_pct_points":float(100*z.minute_drawdown_range.quantile(.9)),"max_intraday_drawdown_range_pct_points":float(100*z.minute_drawdown_range.max()),"days_minute_dd_differs_from_open_by_1pp":int(((z.minute_drawdown_max-z.open_drawdown).abs().ge(.01)|(z.minute_drawdown_min-z.open_drawdown).abs().ge(.01)).sum()),"risk_state_counts":{str(k):int(v) for k,v in z.risk_state.value_counts().sort_index().items()},"by_risk_state":[],"annual":[]}
for state,g in z.groupby("risk_state"):
 metrics["by_risk_state"].append({"risk_state":int(state),"days":len(g),"open_drawdown_min":float(g.open_drawdown.min()),"open_drawdown_max":float(g.open_drawdown.max()),"median_minute_range_pp":float(100*g.minute_drawdown_range.median()),"days_open_to_minute_ge_1pp":int(((g.minute_drawdown_max-g.open_drawdown).abs().ge(.01)|(g.minute_drawdown_min-g.open_drawdown).abs().ge(.01)).sum())})
for year,g in z.groupby("year"):
 metrics["annual"].append({"year":int(year),"days":len(g),"median_range_pp":float(100*g.minute_drawdown_range.median()),"days_open_to_minute_ge_1pp":int(((g.minute_drawdown_max-g.open_drawdown).abs().ge(.01)|(g.minute_drawdown_min-g.open_drawdown).abs().ge(.01)).sum())})
(card/"audit.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(metrics,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_minute_cushion_overlap_20261030",
  "plan_sha256": "d20a68631a4cc8451d60827581dc6f0e70673192ee67463541cbe7fa4ed6619e",
  "days": 558,
  "source_minute_rows": 432720,
  "max_parent_open_drawdown_error": 4.440892098500626e-16,
  "missing_pair_days": 0,
  "recovery_days": 10,
  "minute_bars": 206874,
  "days_with_nonzero_minute_drawdown_range": 552,
  "median_intraday_drawdown_range_pct_points": 1.2218207949221238,
  "p90_intraday_drawdown_range_pct_points": 2.8344031308547457,
  "max_intraday_drawdown_range_pct_points": 7.248087142668869,
  "days_minute_dd_differs_from_open_by_1pp": 267,
  "risk_state_counts": {
    "0": 414,
    "1": 24,
    "2": 60,
    "3": 60
  },
  "by_risk_state": [
    {
      "risk_state": 0,
      "days": 414,
      "open_drawdown_min": 0.0,
      "open_drawdown_max": 0.19269708004252828,
      "median_minute_range_pp": 1.2028024738491738,
      "days_open_to_minute_ge_1pp": 195
    },
    {
      "risk_state": 1,
      "days": 24,
 

## 中文结论

**SOXX：无新候选账户或0/2/5bp门结果。** Qlib分钟价格和逐层点时原生账户在2018–2020年558日闭合开盘回撤，最大误差4.44e−16；206874个开盘后有效分钟可计算当时账户净值/此前高点。552日分钟回撤余量非恒定，267日相对开盘至少偏离1个百分点；日内极差中位1.222个百分点，90分位2.834。风险类别0的414日开盘回撤横跨0至19.27%，同类别不确定连续值。

这只是**新的分钟观测时刻**，不是新独立原始数据：它由实际股数/现金、既有高水位和同一分钟SOXL/SOXX价格确定。b472开盘动作特征原已包含`portfolio_drawdown`；旧固定净值缓冲目标18本账户在SOXX财富/稳定门失败。v5状态控制没有连续回撤，故分钟更新尚未在v5同状态5会话价值任务测试；也不能仅因旧固定目标失败就否定不同接收器。十个恢复日按实际10:32后持仓计算，仅用于描述，不得用事后恢复日期做实时掩码。

2018部分53日、2019完整252日、2020完整253日的日内极差中位约1.246/1.281/1.195个百分点。第一次执行因错误假设半日市16:00收盘在2018-11-23停止，保留失败收据；修为原生账户当日实际终端时间后同一只读范围完成。此卡不读取未来事件价值、2021/2022目标或2023+账户，不涉及新预测、风险投影、整股订单/费用/NAV差。下一步先冻结同状态跨会话动作路径价值和控制，核查连续分钟回撤是否在费用后、按日分组的信息门上有增量；不得改称全新市场来源，也不得复跑旧固定cushion系数。